# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q prophet pandas numpy matplotlib
    print('Packages installed.')

# Time Series Forecasting with Prophet In-Depth

**Description:** This notebook demonstrates time series forecasting using **Facebook Prophet**. Prophet is a procedure for forecasting time series data based on an additive model where non-linear trends are fit with yearly, weekly, and daily seasonality, plus holiday effects. It works best with time series that have strong seasonal effects and several seasons of historical data.

We will cover:
1.  **Synthetic Data Generation**: Creating a time series with trend and seasonality (e.g., website traffic).
2.  **Model Fitting**: Training the Prophet model.
3.  **Forecasting**: Predicting future values.
4.  **Component Analysis**: Decomposing the forecast into trend and seasonal components.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt

# Ensure reproducibility
np.random.seed(42)

## 1. Generate Synthetic Time Series Data
We will simulate daily website traffic over 2 years. 
*   **Trend**: General increase over time.
*   **Seasonality**: Higher traffic on weekdays, lower on weekends.

In [ ]:
date_range = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
n_days = len(date_range)

# 1. Linear Trend
trend = np.linspace(100, 500, n_days)

# 2. Weekly Seasonality (0=Monday, 6=Sunday)
# Weekdays (0-4) get a boost, Weekends (5-6) get a drop
weekly_seasonality = np.array([1.2 if d.weekday() < 5 else 0.8 for d in date_range])

# 3. Noise
noise = np.random.normal(0, 20, n_days)

# Combine
y = (trend * weekly_seasonality) + noise

# Prophet requires columns named 'ds' (Date) and 'y' (Value)
df = pd.DataFrame({'ds': date_range, 'y': y})

df.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['ds'], df['y'])
plt.title('Synthetic Website Traffic (2022-2023)')
plt.xlabel('Date')
plt.ylabel('Visits')
plt.show()

## 2. Fit Prophet Model
Prophet is designed to be easy to use. We simply instantiate the `Prophet` class and call `fit`.

In [ ]:
m = Prophet()
m.fit(df)

## 3. Forecast Future Values
We use `make_future_dataframe` to extend the dates into the future. Let's forecast for the next 365 days (2024).

In [ ]:
future = m.make_future_dataframe(periods=365)
future.tail()

In [ ]:
# Predict
forecast = m.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

## 4. Visualize Forecast and Components
Prophet has built-in plotting tools.
*   **Black dots**: Actual data.
*   **Blue line**: The forecast.
*   **Light blue shade**: Uncertainty interval.

In [ ]:
fig1 = m.plot(forecast)

### Component Plots
This is where Prophet shines for social science. We can decompose the time series into:
*   **Trend**: The overall direction.
*   **Weekly**: Day-of-week effects (look for the weekend drop we simulated).
*   **Yearly**: Annual seasonality (if we had enough data and enabled it).

In [ ]:
fig2 = m.plot_components(forecast)

## Conclusion
Prophet allowed us to model and forecast time series data with minimal tuning. The component plot confirms our simulation: a linear positive trend and a distinct weekly pattern with drops on weekends.